In [18]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import re
import pandas as pd

options = webdriver.ChromeOptions()
# Uncomment the next line if you wish to run Chrome in headless mode
# options.add_argument('--headless')
driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)

driver.get('https://www.fragrantica.com/search/')

perfumes_data = []

try:
    WebDriverWait(driver, 90).until(
        EC.visibility_of_all_elements_located((By.CSS_SELECTOR, 'div.cell.card.fr-news-box'))
    )
    containers = driver.find_elements(By.CSS_SELECTOR, 'div.cell.card.fr-news-box')[:10]  # Limit to first 10 for demonstration
    for i, container in enumerate(containers, start=1):
        perfume_name_element = container.find_element(By.CSS_SELECTOR, 'p > a')
        perfume_brand_element = container.find_element(By.CSS_SELECTOR, 'p > small')
        perfume_name = perfume_name_element.text.strip()
        perfume_brand = perfume_brand_element.text.strip()
        link = perfume_name_element.get_attribute('href')
        
        perfumes_data.append({
            "Rank": i,
            "Perfume Name": perfume_name,
            "Perfume Brand": perfume_brand,
            "Link": link
        })
except Exception as e:
    print("Initial scraping completed or an error occurred:", e)
finally:
    driver.quit()

# Detailed information scraping for each perfume
for perfume in perfumes_data:
    driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)
    try:
        driver.get(perfume['Link'])
        WebDriverWait(driver, 90).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div#main-content')))
        
        description_element = driver.find_element(By.XPATH, '//*[@itemprop="description"]/p')
        description = description_element.text
        year_match = re.search(r'\b\d{4}\b', description)
        perfume['Year'] = year_match.group(0) if year_match else 'Unknown'

        try:
            rating = driver.find_element(By.CSS_SELECTOR, 'span[itemprop="ratingValue"]').text
            votes = driver.find_element(By.CSS_SELECTOR, 'span[itemprop="ratingCount"]').text
        except NoSuchElementException:
            rating, votes = 'Unknown', 'Unknown'
        
        try:
            # Wait for the perfumer section to be loaded
            WebDriverWait(driver, 10).until(EC.visibility_of_all_elements_located(
                (By.XPATH, "//div[contains(@class, 'grid-x grid-padding-x grid-padding-y small-up-2 medium-up-2')]//a")))

            perfumer_elements = driver.find_elements(By.XPATH, "//div[contains(@class, 'grid-x grid-padding-x grid-padding-y small-up-2 medium-up-2')]//a")
            perfumers = ', '.join([elem.text for elem in perfumer_elements if elem.text.strip() != ""])
        except NoSuchElementException:
            perfumers = 'Unknown'
        except TimeoutException:
            perfumers = 'Unknown'


        # Assuming the logic for 'Main Accords' and notes extraction is corrected similarly
        # Scrape the main accords
        try:
            accord_elements = driver.find_elements(By.CSS_SELECTOR, 'div.accord-box > div.accord-bar')
            main_accords = {accord_element.text: accord_element.get_attribute('style').split('width: ')[1].rstrip('%;') for accord_element in accord_elements}
        except NoSuchElementException:
            main_accords = {}

        # Scrape the notes
        notes_xpath = {
            'Top Notes': "//h4[.='Top Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Middle Notes': "//h4[.='Middle Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Base Notes': "//h4[.='Base Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]"
        }
        notes_data = {}
        for note_type, xpath in notes_xpath.items():
            try:
                elements = driver.find_elements(By.XPATH, xpath)
                notes_data[note_type] = [element.text for element in elements if element.text]
            except NoSuchElementException:
                notes_data[note_type] = []

        # Update the perfume dictionary
        perfume.update({
            'Rating': rating,
            'Votes': votes,
            'Perfumers': perfumers,
            'Main Accords': main_accords,
            'Top Notes': notes_data['Top Notes'],
            'Middle Notes': notes_data['Middle Notes'],
            'Base Notes': notes_data['Base Notes']
        })

    except Exception as e:
        print(f"Could not scrape detailed info for {perfume['Perfume Name']}: {e}")
    finally:
        driver.quit()

df = pd.DataFrame(perfumes_data)
df.to_csv('top_10_perfumes_extended.csv', index=False)
print("Extended CSV file has been saved.")

TimeoutException: Message: timeout: Timed out receiving message from renderer: 299.635
  (Session info: chrome=122.0.6261.57)
